# 02 — Operators (Symbols → Behaviour)
### Python Basics · Part 2 of 5

Here is a bug that has cost real companies real money:

```python
19.99 * 100      # you expect 1999
```

Python answers `1998.9999999999998`. Chop that to a whole number — which a lot of billing code does — and a ₹19.99 item is charged as ₹19.98. Multiply by a million orders and somebody is writing a very awkward email.

Operators look like the most boring topic in Python until money, permissions, or a date calculation is involved. This notebook is about the handful of places where they **surprise you**.

## The core idea: an operator is a shortcut, and the *type* decides what it does

`a + b` isn't magic maths — Python asks the object `a` how it wants to be added. That's why `2 + 3` is `5`, `"ab" + "cd"` is `"abcd"`, and `"ab" * 3` is `"ababab"`. Same symbol, different type, different behaviour. (This is also how your own classes will get `+` later, in OOP.)

| Family | Operators | The Iron Gym question it answers |
|---|---|---|
| Arithmetic | `+ - * / // % **` | How many months and days are left? |
| Comparison | `== != < <= > >=` | Is the fee at least ₹1000? |
| Logical | `and or not` | Is she a member **and** not banned? |
| Membership | `in`, `not in` | Is `"pro"` one of our plans? |
| Identity | `is`, `is not` | Is this literally the same object — or `None`? |
| Bitwise | `& \| ^ ~ << >>` | Which permissions are switched on? |
| Assignment | `= += -= *= //= %= **=` | Update this number in place |

## Setup

In [1]:
import math

def show_error(code: str):
    """Run one line and print the result - or the error, without stopping the notebook."""
    try:
        print(f"{code}  ->  {eval(code, globals())!r}")
    except Exception as e:
        print(f"{code}  ->  💥 {type(e).__name__}: {e}")

print("ready")

ready


---
## 1. Arithmetic — three kinds of "divide"

| Operator | Name | `7` and `2` give |
|---|---|---|
| `/` | true division — **always a float** | `3.5` |
| `//` | floor division — rounds **down** | `3` |
| `%` | remainder (modulo) | `1` |
| `**` | power | `49` |

In [2]:
print(7 / 2, 7 // 2, 7 % 2, 7 ** 2)
print(type(8 / 2))                       # / gives a float even when the answer is whole

days_left = 475
months, days = divmod(days_left, 30)     # quotient AND remainder in one call
print(f"{days_left} days = {months} months + {days} days")

seconds = 4000
hours, rest = divmod(seconds, 3600)
minutes, secs = divmod(rest, 60)
print(f"{hours:02d}:{minutes:02d}:{secs:02d}")   # 01:06:40

print((5 + 4) % 7)    # wrap-around: Saturday (5) + 4 days -> day index 2, because % cycles back

3.5 3 1 49
<class 'float'>
475 days = 15 months + 25 days
01:06:40
2


**The negative-number trap.** `//` rounds toward **minus infinity**, not toward zero — and `%` follows it, so the remainder takes the sign of the *divisor*:

In [3]:
print(-7 // 2, -7 % 3)       # -4  and  2
print(int(-7 / 2))           # -3  -> int() truncates toward zero, so it DISAGREES with //

-4 2
-3


---
## 2. The float-money trap

Computers store decimals in binary, and `0.1` has no exact binary form — just like 1/3 has no exact decimal form. Most of the time you never notice. With money you will.

In [4]:
print(0.1 + 0.2)                         # 0.30000000000000004
print(0.1 + 0.2 == 0.3)                  # False
print(math.isclose(0.1 + 0.2, 0.3))      # True -> compare floats with a tolerance, never ==

print(19.99 * 100, int(19.99 * 100))     # 1998.9999999999998  1998   <-- the billing bug
print(round(19.99 * 100))                # 1999 - round first...

price_paise = 1999                       # ...or better: keep money as WHOLE PAISE (an int)
print(f"₹{price_paise / 100:.2f}")       # convert to rupees only when displaying

from decimal import Decimal              # the "exact decimal" tool, for when you need it
print(Decimal("19.99") * 100)

0.30000000000000004
False
True
1998.9999999999998 1998
1999
₹19.99
1999.00


> **Rule:** floats for measurements (BMI, distance), **integers (paise) for money**, `Decimal` when you need exact decimal rules.

---
## 3. Comparison — and `==` vs `is`

`==` asks **"do these have equal value?"**. `is` asks **"are these literally the same object?"** (Remember the name-tag model.)

In [5]:
age = 25
print(18 <= age < 60)                    # chained comparison - reads like maths

print("apple" < "banana", "Zebra" < "apple")   # strings compare by character code: capitals sort first

a = [1, 2]
b = [1, 2]
c = a
print(a == b, a is b)                    # equal values, but two different objects
print(a is c)                            # c is just a second tag on a

x = None
print(x is None)                         # the ONE place you always use `is`

True
True True
True False
True
True


> ⚠️ Small integers (−5 to 256) are cached by CPython, so `a is b` *sometimes* works for numbers by accident. That accident is why people write `if x is 5` — and why it breaks on bigger numbers. **Use `==` for values, `is` only for `None` (and `True`/`False`).**

---
## 4. Logical operators return *values*, not just booleans

`and` / `or` hand back one of their **operands**, and they **short-circuit** — the right side isn't even evaluated if the left already decides the answer.

In [6]:
print(True and False, True or False, not True)

member_name = ""
print(member_name or "Guest")            # `or` returns the first TRUTHY operand -> a default value
print(0 and 99, 5 and 99)                # `and` returns the first FALSY operand, else the last one

is_member = False
print(is_member and (1 / 0))             # no ZeroDivisionError - the right side never ran
print(True or print("never printed"))    # same idea

False True False
Guest
0 99
False
True


Precedence among the three: **`not` → `and` → `or`**. When in doubt, add parentheses — nobody has ever been angry at a readable bracket.

In [7]:
print(True or False and False)           # and binds tighter: True or (False and False) -> True
print((True or False) and False)         # brackets change the meaning -> False

True
False


---
## 5. Membership — "is it one of these?"

In [8]:
plan = "pro"
print(plan in ("basic", "pro", "elite"))          # beats three `==` joined by `or`
print("rah" in "Rahul", "x" not in "Rahul")        # substring test on strings
print(5 in range(1, 10), 10 in range(1, 10))       # range stop is exclusive

True
False True
True False


---
## 6. Assignment operators — and a list surprise

`x += 1` is shorthand for `x = x + 1` — **except on lists**, where `+=` changes the list *in place* while `+` builds a new one. Name-tag model again:

In [9]:
visits = 0
visits += 1; visits *= 5; visits -= 2; visits **= 2; visits //= 3; visits %= 4
print(visits)                            # trace it by hand: 1 -> 5 -> 3 -> 9 -> 3 -> 3

a = [1, 2]
b = a
a += [3]                                 # in place: b sees it
print(b)
a = a + [4]                              # builds a NEW list; the 'a' tag moves, b is unchanged
print(a, b)

3
[1, 2, 3]
[1, 2, 3, 4] [1, 2, 3]


---
## 7. Bitwise operators — the real-world use: permission flags

You will rarely do bit-twiddling for fun, but **flags** are everywhere: file permissions, feature toggles, game states. Each permission is one bit, and a member's access is a single integer.

| Operator | Does | Use for |
|---|---|---|
| `a \| b` | OR | **grant** — switch bits on |
| `a & b` | AND | **check** — is this bit on? |
| `a & ~b` | AND with inverse | **revoke** — switch a bit off |
| `a ^ b` | XOR | **toggle** |
| `1 << n` | shift left | `2**n` |

In [10]:
GYM, POOL, TRAINER, STEAM = 1, 2, 4, 8            # one bit each: 0001, 0010, 0100, 1000

member = GYM | TRAINER                             # grant two permissions
print(bin(member), member)                         # 0b101  5

print("has pool?   ", bool(member & POOL))         # check
print("has trainer?", bool(member & TRAINER))

member |= POOL                                     # grant pool
member &= ~TRAINER                                 # revoke trainer
member ^= STEAM                                    # toggle steam (was off -> on)
print(bin(member), member)                         # 0b1011  11
print(1 << 3, 16 >> 2)                             # 8  4

0b101 5
has pool?    False
has trainer? True
0b1011 11
8 4


---
## 8. Precedence — who goes first

From **highest** to lowest: `**` → unary `-`/`~` → `* / // %` → `+ -` → shifts → `&` → `^` → `\|` → comparisons / `in` / `is` → `not` → `and` → `or`.

In [11]:
print(2 + 3 * 4 ** 2 / 8)     # 4**2=16 -> 3*16=48 -> 48/8=6.0 -> 2+6.0
print(-2 ** 2)                # -4 !  ** binds tighter than the unary minus
print((-2) ** 2)              # 4

8.0
-4
4


---
## Real-world capstone: Iron Gym fee calculator (in whole paise)

Everything from this notebook in one place: integer money, `//` for percentages, `min`/`max` for a capped late fine, and `%`/`divmod` for display. Rules at Iron Gym: 18% GST on the discounted amount; **3 grace days**, then **₹10 per late day**, capped at **₹500**.

In [12]:
GST_PERCENT = 18
GRACE_DAYS, FINE_PER_DAY, FINE_CAP = 3, 1000, 50000      # all in PAISE

monthly_fee = 149950          # ₹1,499.50
months = 6
discount_percent = 10
days_late = 9

subtotal = monthly_fee * months
discount = subtotal * discount_percent // 100
taxable  = subtotal - discount
gst      = taxable * GST_PERCENT // 100
fine     = min(max(days_late - GRACE_DAYS, 0) * FINE_PER_DAY, FINE_CAP)
total    = taxable + gst + fine

def rupees(paise):                       # tiny helper: a named, reusable block (functions: Day 10)
    sign = "-" if paise < 0 else ""
    whole, part = divmod(abs(paise), 100)   # abs() first - see the note below the receipt
    return f"{sign}₹{whole:,}.{part:02d}"

print(f"{'Subtotal':<12}{rupees(subtotal):>14}")
print(f"{'Discount':<12}{rupees(-discount):>14}")
print(f"{'GST 18%':<12}{rupees(gst):>14}")
print(f"{'Late fine':<12}{rupees(fine):>14}")
print(f"{'':-<26}")
print(f"{'TOTAL':<12}{rupees(total):>14}")

assert total == 961481, "the receipt total changed - check your arithmetic"

Subtotal         ₹8,997.00
Discount          -₹899.70
GST 18%          ₹1,457.51
Late fine           ₹60.00
--------------------------
TOTAL            ₹9,614.81


Notice there isn't a single float above, so there's nothing to drift. Everything was `int`, and `rupees()` is the *only* place a decimal point appears.

> 🐛 **A bug the first draft of this notebook actually had:** `rupees()` originally did `paise // 100` and `paise % 100` directly. For the discount (`-89970` paise) that printed `₹ -900.30` instead of `-899.70`, because `-89970 // 100` is `-900` — the negative floor-division trap from section 1, biting inside a "simple" formatter. The fix is to take `abs()` first and carry the sign separately. Operators you "know" still need testing on negative numbers.

### Try it yourself
1. Set `days_late = 30`. Does the fine stop at ₹500? Remove `min(...)` and watch it run away.
2. Make the discount **12.5%** while keeping everything an integer (hint: multiply by `125` and divide by `1000`).
3. Reproduce the billing bug on purpose: redo the whole calculator with floats (`1499.50`, `0.18`) and compare the total to the integer version. How far apart are they after 1,000 members?
4. Extend the permission flags: add `LOCKER = 16`. Write one line that gives a member **every** permission (hint: `GYM | POOL | ...`), and one line that checks they have **all** of `POOL | STEAM` at once (`(m & mask) == mask`).
5. Predict, then run: `not 0 or 1 and 2`, `2 ** 3 ** 2`, `-7 // 2 * 2 + -7 % 2`.

---
## Quick reference

| I want to… | Use |
|---|---|
| quotient + remainder together | `divmod(a, b)` |
| compare floats | `math.isclose(a, b)` — never `==` |
| store money | whole paise (`int`), format only when displaying |
| default when empty | `value or "default"` |
| "one of these?" | `x in (a, b, c)` |
| test for `None` | `x is None` |
| grant / check / revoke / toggle a flag | `\|=` / `&` / `&= ~` / `^=` |

**Rule of thumb:** *money = integers, flags = bits, defaults = `or`, `None` = `is`.*

---
## 🎯 Interview corner

**1. `/` vs `//` vs `%`?** True division (always float), floor division (rounds toward −∞), remainder.
**2. `==` vs `is`?** Equal value vs same object. Use `is` only for `None`/`True`/`False`.
**3. Why is `0.1 + 0.2 != 0.3`?** Binary floating point can't represent 0.1 exactly. Use `math.isclose`, `Decimal`, or integers.
**4. What does `a or b` return?** The first truthy operand (or the last operand) — not necessarily a `bool`.
**5. What is short-circuiting?** In `a and b` / `a or b`, `b` is only evaluated if `a` doesn't already decide the result.
**6. `x += 1` vs `x = x + 1`?** Same for numbers; for lists `+=` mutates in place.

**Next:** `03_conditionals/` — turning those yes/no answers into decisions.